# Data Collection for Pitcher Strikeout Prediction

This notebook explores the data collection process using the `mlb_data` package.

Data sources:
- **Pitcher game logs**: Daily pitching stats from Statcast (100% accurate team IDs)
- **Team batting stats**: Season batting statistics for opponent analysis
- **Pitcher season stats**: Season-level pitching statistics

In [ ]:
from datetime import datetime, timedelta

import pandas as pd

# Import from mlb_data package
from mlb_data import (
    get_pitcher_game_logs,
    get_pitcher_season_stats,
    get_team_batting,
    ALL_TEAMS,
    list_teams,
)

# Settings
SEASON = 2024
pd.set_option('display.max_columns', 50)

## Team Abbreviations

All 30 MLB teams with unambiguous abbreviations:

In [ ]:
list_teams()

## Team Batting Stats

Season-level batting statistics for each team. Used to assess opponent difficulty.

In [ ]:
team_batting = get_team_batting(SEASON)
print(f"Shape: {team_batting.shape}")
team_batting.head(10)

In [ ]:
# Available columns
print("Team batting columns:")
print(team_batting.columns.tolist())

## Pitcher Game Logs

Daily pitching statistics from Statcast. This is the primary data source with accurate team identification.

Key columns:
- `pitcher_id`: Unique MLB player ID
- `Name`: Pitcher name
- `team_abbrev`: Pitcher's team (e.g., NYY, LAD)
- `opp_abbrev`: Opponent team (e.g., BOS, CHC)
- `SO`: Strikeouts (our target variable)
- `IP`: Innings pitched
- `pitches`: Total pitches thrown

In [ ]:
# Collect pitcher game logs for a date range
# Note: This uses Statcast and may take a minute for large date ranges

start_date = f"{SEASON}-04-01"
end_date = f"{SEASON}-06-01"  # Adjust as needed

pitcher_games = get_pitcher_game_logs(
    start_date=start_date,
    end_date=end_date,
    starters_only=True,  # Only starting pitchers
)

print(f"Shape: {pitcher_games.shape}")
print(f"Date range: {pitcher_games['game_date'].min()} to {pitcher_games['game_date'].max()}")
pitcher_games.head(10)

In [ ]:
# Available columns
print("Pitcher game log columns:")
print(pitcher_games.columns.tolist())

In [ ]:
# Verify accurate team identification
# No more ambiguous city names - we can distinguish NYY from NYM, CHC from CHW, etc.

print("Unique teams in data:")
print(sorted(pitcher_games['team_abbrev'].unique()))

print("\nGames by team:")
pitcher_games['team_abbrev'].value_counts().head(10)

In [ ]:
# Strikeout distribution (our target variable)
print("Strikeout statistics:")
print(pitcher_games['SO'].describe())

print("\nStrikeout distribution:")
pitcher_games['SO'].value_counts().sort_index()

## Pitcher Season Stats

Season-level pitching statistics from FanGraphs. Useful for baseline pitcher quality metrics.

In [ ]:
pitcher_season = get_pitcher_season_stats(SEASON, qual=0)
print(f"Shape: {pitcher_season.shape}")
pitcher_season.head(10)

In [ ]:
# Available columns
print("Pitcher season stats columns:")
print(pitcher_season.columns.tolist())

## Save Data

Save the collected data to CSV files for use in feature engineering.

In [ ]:
from pathlib import Path

# Create output directory
output_dir = Path("../data/raw")
output_dir.mkdir(parents=True, exist_ok=True)

# Save to CSV
team_batting.to_csv(output_dir / "team_batting.csv", index=False)
pitcher_season.to_csv(output_dir / "pitcher_season.csv", index=False)
pitcher_games.to_csv(output_dir / "pitcher_games.csv", index=False)

print(f"Saved data to {output_dir.resolve()}")
print(f"  - team_batting.csv: {len(team_batting)} rows")
print(f"  - pitcher_season.csv: {len(pitcher_season)} rows")
print(f"  - pitcher_games.csv: {len(pitcher_games)} rows")